# docling PDF loader — /G glyph decode

Some PDFs use custom glyph-encoded fonts where docling emits `/G44/G49/G52…` sequences.
These turn out to be plain **ASCII hex** — `/G44` = `chr(0x44)` = `'D'`.
We decode them in-place with a regex substitution: instant, no OCR needed.

This notebook:
1. Shows the raw /G corruption from docling
2. Verifies the decode function produces readable text
3. Runs `extract_text_from_pdf` on both a corrupt and a clean PDF
4. Sanity-checks that chunking still works

In [ ]:
!pip install docling --quiet

In [10]:
import sys, re, logging
from pathlib import Path
from types import SimpleNamespace

sys.path.insert(0, "..")
logging.basicConfig(level=logging.INFO)

PDF_DIR = Path("../data/train/pdf_train")

# Known corrupted PDF (glyph-encoded fonts → /G sequences from docling)
CORRUPTED_PDF = PDF_DIR / "afe620b9beac86c1027b96d31d396407.pdf"

# Pick any other PDF as a clean baseline
CLEAN_PDF = next(p for p in sorted(PDF_DIR.glob("*.pdf")) if p != CORRUPTED_PDF)

print(f"Corrupted : {CORRUPTED_PDF.name}")
print(f"Clean     : {CLEAN_PDF.name}")

Corrupted : afe620b9beac86c1027b96d31d396407.pdf
Clean     : 05-03-18-political-release.pdf


## 1. Demonstrate the corruption

Run docling directly on the corrupted PDF and show that the output is glyph-encoded garbage.

In [4]:
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import DocItemLabel

converter = DocumentConverter()
result = converter.convert(str(CORRUPTED_PDF))

print("Conversion status:", result.status)
print("\n--- First 500 chars of markdown export ---")
print(result.document.export_to_markdown()[:500])

INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash 43cb18089737123a0a3edf8fc85ce8d9
INFO:docling.models.stages.ocr.auto_ocr_model:Auto OCR model selected ocrmac.
INFO:docling.utils.accelerator_utils:Accelerator device: 'mps'
INFO:docling.utils.accelerator_utils:Accelerator device: 'mps'
INFO:docling.pipeline.base_pipeline:Processing document afe620b9beac86c1027b96d31d396407.pdf
INFO:docling.document_converter:Finished converting document afe620b9beac86c1027b96d31d396407.pdf in 144.44 sec.


Conversion status: ConversionStatus.SUCCESS

--- First 500 chars of markdown export ---
## /G44/G49/G52/G45/G43/G54/G4F/G52/G53/G27/G20/G52/G45/G50/G4F/G52/G54/G20/G26/G20/G4D/G41/G4E/G41/G47/G45/G4D/G45/G4E/G54/G20/G44/G49/G53/G43/G55/G53/G53/G49/G4F/G4E/G20/G41/G4E/G44/G20/G41/G4E/G41/G4C/G59/G53/G49/G53

/G59 /G6F/G75/G72 /G44/G69/G72/G65/G63/G74/G6F/G72/G73 /G68/G61/G76/G65 /G70/G6C/G65/G61/G73/G75/G72/G65 /G69/G6E /G73/G75/G62/G6D/G69/G74/G74/G69/G6E/G67 /G74/G68/G65/G69/G72 /G41/G6E/G6E/G75/G61/G6C /G52/G65/G70/G6F/G72/G74 /G61/G6E/G64 /G41/G63/G63/G6F/G75/G6E/G74/G73 /G6F/G6


## 2. Decode /G sequences — ASCII hex

`/G44` → `chr(0x44)` = `'D'`, `/G49` → `'I'`, `/G52` → `'R'` … just ASCII.

In [5]:
import re
from preprocessing.pdf_loader import _decode_glyph_text

SKIP_LABELS_SET = {DocItemLabel.PAGE_FOOTER, DocItemLabel.PAGE_HEADER}

raw_blocks = []
for item, _ in result.document.iterate_items():
    if item.label in SKIP_LABELS_SET:
        continue
    text = item.text.strip() if hasattr(item, 'text') and item.text else ''
    if text:
        raw_blocks.append(text)

print(f'Total blocks: {len(raw_blocks)}')
print('\n--- Raw (corrupted) ---')
print(raw_blocks[0][:120])
print('\n--- Decoded ---')
print(_decode_glyph_text(raw_blocks[0])[:120])

Total blocks   : 268
Corrupted      : 74 (28%)
Detector fires : True

Sample corrupted block:
/G44/G49/G52/G45/G43/G54/G4F/G52/G53/G27/G20/G52/G45/G50/G4F/G52/G54/G20/G26/G20/G4D/G41/G4E/G41/G47/G45/G4D/G45/G4E/G54


## 3. `extract_text_from_pdf` — decode is applied automatically

In [11]:
from preprocessing.pdf_loader import extract_text_from_pdf

out_corrupted = extract_text_from_pdf(CORRUPTED_PDF)
print(f'[corrupted] blocks: {len(out_corrupted["blocks"])}')
print('First 3 blocks:')
for b in out_corrupted['blocks'][:3]:
    print(f"  [p{b['page_number']}] {b['text'][:100]!r}")

print()

out_clean = extract_text_from_pdf(CLEAN_PDF)
print(f'[clean]     blocks: {len(out_clean["blocks"])}')
print('First 3 blocks:')
for b in out_clean['blocks'][:3]:
    print(f"  [p{b['page_number']}] [{b['category']}] {b['text'][:100]!r}")

INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash 43cb18089737123a0a3edf8fc85ce8d9
INFO:docling.models.stages.ocr.auto_ocr_model:Auto OCR model selected ocrmac.
INFO:docling.utils.accelerator_utils:Accelerator device: 'mps'
INFO:docling.utils.accelerator_utils:Accelerator device: 'mps'
INFO:docling.pipeline.base_pipeline:Processing document afe620b9beac86c1027b96d31d396407.pdf
INFO:docling.document_converter:Finished converting document afe620b9beac86c1027b96d31d396407.pdf in 156.77 sec.


Forced-OCR blocks : 268

First 5 blocks:
  [p1] '/G44/G49/G52/G45/G43/G54/G4F/G52/G53/G27/G20/G52/G45/G50/G4F/G52/G54/G20/G26/G20/G4D/G41/G4E/G41/G47'
  [p1] '/G59 /G6F/G75/G72 /G44/G69/G72/G65/G63/G74/G6F/G72/G73 /G68/G61/G76/G65 /G70/G6C/G65/G61/G73/G75/G72'
  [p2] '/G54/G48/G45/G20/G45/G58/G54/G45/G4E/G54/G20/G4F/G46._/G20/G54/G48/G45/G20/G44/G49/G53/G43/G52/G49/G'
  [p2] '/G43/G6C/G65/G61/G72/G6C/G79 /G74/G68/G65 /G74/G61/G78 /G70/G6F/G6C/G69/G63/G79 /G6F/G66 /G70/G6C/G6'
  [p2] '/G41 /G6D/G6F/G72/G65 /G65/G71/G75/G69/G74/G61/G62/G6C/G65 /G74/G61/G78 /G62/G61/G73/G65 /G6D/G75/G7'


## 4. Chunking sanity check

In [7]:
from preprocessing.pdf_chunker import Chunking
from types import SimpleNamespace

def to_ns(doc_out):
    return [
        SimpleNamespace(category=b['category'], text=b['text'], page_number=b['page_number'])
        for b in doc_out['blocks']
    ]

for label, doc_out in [('corrupted (decoded)', out_corrupted), ('clean', out_clean)]:
    chunks = Chunking(to_ns(doc_out)).fixed_size(max_chars=1000)
    print(f'{label}: {len(chunks)} chunks')
    print(f'  first chunk: {chunks[0]["text"][:120]!r}')
    print()

INFO:preprocessing.pdf_loader:Initialising docling DocumentConverter...
INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash 027d654c940dc566b51eee6c3c47977d
INFO:docling.models.stages.ocr.auto_ocr_model:Auto OCR model selected ocrmac.
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:docling.pipeline.base_pipeline:Processing document afe620b9beac86c1027b96d31d396407.pdf
INFO:docling.document_converter:Finished converting document afe620b9beac86c1027b96d31d396407.pdf in 171.67 sec.
INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.pipeline.base_pipeline:Processing document 05-03-18-political-release.pdf


[corrupted] blocks: 413
First 3 blocks:
  [p1] '\x01\x02\x03\x04 \x05\x06\x04\x07\x08\t\x02\x04\n \x0b\x0c\r\x07 \x0e\x0f\x07\x0c\n\x03\x04\x07 \x06\x10 \n\x03\x11\x12\x06\t\t\x06\x10\x13 \t\x0b\x07\x06\x04 \x14\x10\x10\x03\x0c\x0f \x15\x07\x0e\x02\x04\t \x0c\x10\x16 \x14\x08\x08\x02\x03\x10\t\n \x02\x17 \t\x0b\x07 \x18\x02\x12\x0e\x0c\x10\x19 \x17\x02\x04 \t\x0b\x07 \x19'
  [p1] '\x07\x10\x16\x07\x16 \x1a\x1b\n\t \x1c\x0c\x04\x08\x0b \x1d\x1e\x1e\x1a'
  [p1] '\x14\x13\x0c\x06\x10\n\t \x0c \x17\x02\x04\x07\x08\x0c\n\t  \x05! \x13\x04\x02"\t\x0b \x02\x17 #\x1f$%& \'\x10\x16\x06\x0c \x0c\x08\x0b\x06\x07\r\x07\x16 \x0c  \x05! \x13\x04\x02"\t\x0b \x02\x17 (\x1f\x1a%\x1f \'\x10 \t\x0b\x07 \x17\x06\x04\n\t \t"\x02 )\x03\x0c\x04\t\x07\x04'



INFO:docling.document_converter:Finished converting document 05-03-18-political-release.pdf in 41.15 sec.


[clean]     blocks: 122
First 3 blocks:
  [p1] [NarrativeText] 'Negatively on Issues, but'
  [p1] [NarrativeText] 'Critical of His Conduct'
  [p1] [NarrativeText] 'FOR RELEASE MAY 3, 2018'
